In [ ]:
# Import required libraries
import sys
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Add acorn to path
SCRIPT_DIR = Path.cwd()
WORKSPACE_ROOT = SCRIPT_DIR.parent
sys.path.insert(0, str(WORKSPACE_ROOT / 'acorn'))

# Import directly to avoid circular dependency issue
from acorn.stages.graph_construction.models import metric_learning
MetricLearning = metric_learning.MetricLearning

print("Imports successful!")

: 

In [8]:


def load_model(checkpoint_name):
    """
    Load trained metric learning model from checkpoint
    
    Args:
        checkpoint_name: Name of checkpoint (with or without .ckpt extension)
    
    Returns:
        model: Loaded MetricLearning model in eval mode
        hparams: Model hyperparameters
    """
    # Add .ckpt extension if needed
    if not checkpoint_name.endswith('.ckpt'):
        checkpoint_name = checkpoint_name + '.ckpt'
    
    checkpoint_path = SCRIPT_DIR / 'saved_models' / checkpoint_name
    
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    
    print(f"Loading model from: {checkpoint_path}")
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    # Extract hyperparameters
    hparams = checkpoint['hyper_parameters']
    
    print(f"  Model architecture:")
    print(f"    Input features: {hparams['node_features']}")
    print(f"    Hidden layers: {hparams['nb_layer']} x {hparams['emb_hidden']}")
    print(f"    Embedding dim: {hparams['emb_dim']}D")
    print(f"    Activation: {hparams['activation']}")
    
    # Create model instance and load weights
    model = MetricLearning(hparams)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    
    return model, hparams

# Load the model
model, hparams = load_model('latentmodel_val_loss=0.0034.ckpt')  # Change this to your model name

Loading model from: /data/alice/bkuipers/low_pt_gnn_pipeline/saved_models/latentmodel_val_loss=0.0034.ckpt
  Model architecture:
    Input features: ['hit_r', 'hit_phi', 'hit_z']
    Hidden layers: 4 x 64
    Embedding dim: 8D
    Activation: Tanh


In [10]:
# Load a graph from feature_store
dataset = 'trainset'  # trainse or 'valset', 'testset'
event_idx = 0  # Choose which event to analyze

graph_path = SCRIPT_DIR / 'data' / 'feature_store' / dataset / f'event{event_idx:09d}-graph.pyg'
graph = torch.load(graph_path)

# Load truth information
truth_csv_path = str(graph_path).replace('-graph.pyg', '-truth.csv')
truth_df = pd.read_csv(truth_csv_path)

# Add cylindrical coordinates to graph
x = truth_df['x'].values
y = truth_df['y'].values
z_cart = truth_df['z'].values

r = np.sqrt(x**2 + y**2)
phi = np.arctan2(y, x)

graph.hit_r = torch.tensor(r, dtype=torch.float32)
graph.hit_phi = torch.tensor(phi, dtype=torch.float32)
graph.hit_z = torch.tensor(z_cart, dtype=torch.float32)
graph.particle_id = torch.tensor(truth_df['particle_id'].values, dtype=torch.long)

print(f"Loaded graph: {graph.num_nodes} hits")
print(f"Unique particles: {len(graph.particle_id.unique()) - 1}")  # -1 for noise

Loaded graph: None hits
Unique particles: 19


/data/alice/bkuipers/miniconda3/envs/acorn/lib/python3.10/site-packages/torch_geometric/data/data.py:187: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'hit_r', 'hit_particle_id', 'particle_id', 'track_particle_pt', 'hit_region', 'track_particle_id', 'hit_id', 'hit_x', 'hit_z', 'hit_y', 'track_edges', 'config', 'hit_phi', 'event_id', 'track_particle_nhits', 'track_particle_eta', 'hit_eta', 'track_particle_radius'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  return sum([v.num_nodes for v in self.node_stores])


In [11]:
def map_to_latent_space(model, graph, node_features):
    """
    Map graph hits to learned latent space
    
    Args:
        model: Trained MetricLearning model
        graph: PyG graph with node features
        node_features: List of feature names (e.g., ['hit_r', 'hit_phi', 'hit_z'])
    
    Returns:
        embeddings: Tensor of shape [num_hits, emb_dim]
    """
    # Extract features from graph
    feature_list = []
    for feat_name in node_features:
        if hasattr(graph, feat_name):
            feature_list.append(getattr(graph, feat_name))
        else:
            raise ValueError(f"Feature {feat_name} not found in graph")
    
    # Stack features into tensor [num_hits, num_features]
    x = torch.stack(feature_list, dim=1).float()
    
    # Get embeddings
    with torch.no_grad():
        embeddings = model(x)
    
    return embeddings

# Map hits to latent space
embeddings = map_to_latent_space(model, graph, hparams['node_features'])

print(f"Embeddings shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}D")

Embeddings shape: torch.Size([248, 8])
Embedding dimension: 8D


In [12]:
# Visualize latent space with PCA to 2D
from sklearn.decomposition import PCA

emb_np = embeddings.cpu().detach().numpy()
particle_ids = graph.particle_id.cpu().detach().numpy()

# Apply PCA to reduce to 2 dimensions
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(emb_np)

#print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%} (eigen value fraction)")

# Create 2D scatter plot
fig, ax = plt.subplots(figsize=(14, 10))

# Color by particle ID (limit to a few particles for clarity)
unique_pids = np.unique(particle_ids)
unique_pids = unique_pids[unique_pids > 0]  # Remove noise

# Plot each particle with different color
import matplotlib.cm as cm
colors = cm.rainbow(np.linspace(0, 1, min(len(unique_pids), 20)))

for i, pid in enumerate(unique_pids[:20]):  # Plot first 20 particles
    mask = particle_ids == pid
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], 
               c=[colors[i]], label=f'Particle {pid}', s=50, alpha=0.7)

# Plot noise in gray
noise_mask = particle_ids == 0
if noise_mask.any():
    ax.scatter(emb_2d[noise_mask, 0], emb_2d[noise_mask, 1],
               c='gray', label='Noise', s=20, alpha=0.3)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title(f'Latent Space Embedding - PCA Projection to 2D (from {embeddings.shape[1]}D)')
#ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=2)
ax.grid(True)

plt.tight_layout()
plt.show()

print("\nHits from the same particle should cluster together in latent space!")

RuntimeError: Numpy is not available

In [ ]:
# Visualize how a radius in 8D latent space projects to 2D PCA space
# Pick a query node and show neighbors within radius

query_idx = 50  # Choose a hit to analyze
radius_8d = 0.15  # Radius in 8D latent space (normalized, same as r_train)

# Get query point embedding
query_emb_8d = embeddings[query_idx].cpu().detach().numpy()
query_pid = graph.particle_id[query_idx].item()

# Find neighbors within radius in 8D space
distances_8d = np.linalg.norm(emb_np - query_emb_8d, axis=1)
neighbors_mask = distances_8d < radius_8d

print(f"Query hit: index={query_idx}, particle_id={query_pid}")
print(f"Neighbors within r={radius_8d} in 8D: {neighbors_mask.sum()}")
print(f"Same particle: {(graph.particle_id.cpu().numpy()[neighbors_mask] == query_pid).sum()}")
print(f"Different particles: {(graph.particle_id.cpu().numpy()[neighbors_mask] != query_pid).sum()}")

# Sample points on 8D sphere surface around query to visualize boundary
n_samples = 1000
sphere_points_8d = np.random.randn(n_samples, 8)  # Random directions
sphere_points_8d = sphere_points_8d / np.linalg.norm(sphere_points_8d, axis=1, keepdims=True)  # Normalize
sphere_points_8d = query_emb_8d + radius_8d * sphere_points_8d  # Scale and translate

# Project sphere points to 2D PCA space
sphere_points_2d = pca.transform(sphere_points_8d)
query_2d = emb_2d[query_idx]

# Create visualization
fig, ax = plt.subplots(figsize=(12, 10))

# Plot all hits in gray
ax.scatter(emb_2d[:, 0], emb_2d[:, 1], c='lightgray', s=20, alpha=0.3, label='All hits')

# Plot neighbors within radius
neighbor_indices = np.where(neighbors_mask)[0]
same_particle_mask = graph.particle_id.cpu().numpy()[neighbor_indices] == query_pid
diff_particle_mask = graph.particle_id.cpu().numpy()[neighbor_indices] != query_pid

ax.scatter(emb_2d[neighbor_indices[same_particle_mask], 0], 
           emb_2d[neighbor_indices[same_particle_mask], 1],
           c='green', s=60, alpha=0.7, label=f'Same particle (pid={query_pid})', edgecolors='darkgreen')

ax.scatter(emb_2d[neighbor_indices[diff_particle_mask], 0], 
           emb_2d[neighbor_indices[diff_particle_mask], 1],
           c='red', s=60, alpha=0.7, label='Different particles', edgecolors='darkred')

# Plot the 8D sphere boundary projected to 2D (will be an ellipse)
ax.scatter(sphere_points_2d[:, 0], sphere_points_2d[:, 1], 
           c='blue', s=1, alpha=0.2, label=f'8D sphere boundary (r={radius_8d})')

# Plot query point
ax.scatter(query_2d[0], query_2d[1], c='yellow', s=200, marker='*', 
           edgecolors='black', linewidths=2, label='Query hit', zorder=10)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title(f'Radius Cut in 8D Latent Space → Ellipse in 2D PCA Space')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.axis('equal')  # Equal aspect ratio to see true ellipse shape

plt.tight_layout()
plt.show()

print("\nThe blue cloud shows the 8D sphere boundary projected to 2D - it's an ellipse!")
print("Green = true edges (same particle), Red = false edges (different particles)")

NameError: name 'embeddings' is not defined

In [ ]:
# Compute pairwise distances in latent space (sample for efficiency)
sample_size = min(1000, len(embeddings))
sample_indices = np.random.choice(len(embeddings), sample_size, replace=False)

emb_sample = embeddings[sample_indices]
pid_sample = graph.particle_id[sample_indices]

# Compute distances
from scipy.spatial.distance import cdist
distances = cdist(emb_sample.cpu().detach().numpy(), emb_sample.cpu().detach().numpy(), metric='euclidean')

# Separate same-particle vs different-particle distances
same_particle_dists = []
diff_particle_dists = []

for i in range(len(pid_sample)):
    for j in range(i+1, len(pid_sample)):
        if pid_sample[i] > 0 and pid_sample[j] > 0:  # Ignore noise
            if pid_sample[i] == pid_sample[j]:
                same_particle_dists.append(distances[i, j])
            else:
                diff_particle_dists.append(distances[i, j])

# Plot histogram
plt.figure(figsize=(10, 6))
plt.hist(same_particle_dists, bins=50, alpha=0.6, label='Same particle', color='green')
plt.hist(diff_particle_dists, bins=50, alpha=0.6, label='Different particle', color='red')
plt.xlabel('Distance in latent space')
plt.ylabel('Count')
plt.title('Distance Distribution: Same vs Different Particles')
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Mean distance (same particle): {np.mean(same_particle_dists):.4f}")
print(f"Mean distance (different particle): {np.mean(diff_particle_dists):.4f}")
print(f"Ratio: {np.mean(diff_particle_dists) / np.mean(same_particle_dists):.2f}x")
print("\nGood model: different particle distances >> same particle distances")

RuntimeError: Could not infer dtype of numpy.int64